In [56]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [57]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [58]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 19:41:41, wtch_dt_end:2026-07-21 19:41:41


In [59]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [60]:
@file:DependsOn("org.json:json:20250107")

In [61]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [62]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Number,2133,1147,0,0.260000,61,5.211476,2.560636,0.250000,3.944000,5.510000,6.430000,19.907000
rtmWqChpla,Comparable<*>,2133,1316,0,,180,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2133,1,0,,2133,null,null,,,,,
rtmWqWtchStaCd,String,2133,14,0,SEA1005,180,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2133,2133,0,1,1,1067.000000,615.888383,1,533.666667,1067.000000,1600.333333,2133
rtmWqTu,Int,2133,160,0,5,194,25.778715,34.192834,0,5.000000,11.000000,32.000000,232
ph,Double,2133,144,0,7.520000,53,7.679090,0.304409,7.020000,7.470000,7.630000,7.920000,9.080000
rtmWqSlnty,Number,2133,1974,0,32.705002,4,21.848645,10.277931,0.020000,13.959000,26.649000,29.576000,34.032001
rtmWqCndctv,Float,2133,2046,0,44.272999,3,34.113092,15.372898,0.046000,23.254666,39.889999,45.016333,54.451000
rtmWqWtchDtlDt,String,2133,189,0,2026-07-20 20:10:00.0,14,null,null,2026-07-20 19:45:00.0,2026-07-21 01:40:00.0,2026-07-21 07:40:00.0,2026-07-21 13:30:00.0,2026-07-21 19:15:00.0


In [63]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) "0" else String.format("%.2f", value.toFloat())

}.convert { rtmWqDoxn and  rtmWqTu and rtmWqSlnty and rtmWqCndctv and rtmWtchWtem }.with { String.format("%.2f", it.toFloat()) }


df.schema()

rtmWqDoxn: String
rtmWqChpla: String
rtmWqWtchStaCd: String
num: Int
rtmWqTu: String
ph: Double
rtmWqSlnty: String
rtmWqCndctv: String
rtmWqWtchDtlDt: LocalDateTime
rtmWtchWtem: String

In [64]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [65]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1,2026-07-20T19:45,4.78,7.57,SEA5003,13.00,7.520000,32.58,49.29,24.49
2,2026-07-20T19:45,0.85,1.78,SEA5002,51.00,7.320000,14.86,24.59,29.65
3,2026-07-20T19:45,3.43,3.37,SEA2007,35.00,7.890000,32.13,51.89,24.55
4,2026-07-20T19:45,4.78,0.92,NEP3001,8.00,7.840000,25.18,39.51,25.94
5,2026-07-20T19:45,7.37,17.25,SEA6001,49.00,8.030000,29.31,45.37,27.21


In [66]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .convert{수온}.toDouble()
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="3Rq3xh" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("3Rq3xh");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.7845794E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.78458E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845803E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845809E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845812E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845818E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845821E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.7845827E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.784583E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845836E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845839E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845845E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845848E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845854E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845857E12,1.7845863E12,1.7845863E12,1.7845863E12,1.7845863E12,1.7845863E